In [2]:
import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta
from tqdm import tqdm

# Function to generate a random date between two dates
def random_date(start, end, size):
    return start + pd.to_timedelta(np.random.randint(0, int((end - start).days), size=size), unit='D')

# Load base tables
uscities = pd.read_csv('uscities.csv')
auto_make_model = pd.read_csv('AUTO_MAKE_MODEL.csv')
auto_insurance_agent_master = pd.read_csv('AUTO_INSURANCE_AGENT_MASTER.csv')
auto_insurance_policy_master = pd.read_csv('AUTO_INSURANCE_POLICY_MASTER.csv')

# Convert AGENT_INACTIVE_FROM to datetime
auto_insurance_agent_master['AGENT_INACTIVE_FROM'] = pd.to_datetime(auto_insurance_agent_master['AGENT_INACTIVE_FROM'], errors='coerce')

# Define date range for APPLICATION_DATE
start_date = datetime.strptime('2015-01-01', '%Y-%m-%d')
end_date = datetime.strptime('2024-12-31', '%Y-%m-%d')

# Precompute some values to avoid repetitive calculations
agent_ids = auto_insurance_agent_master[auto_insurance_agent_master['AGENT_INACTIVE_FROM'].isnull() | (auto_insurance_agent_master['AGENT_INACTIVE_FROM'] < end_date)]['AGENT_ID'].tolist()
texas_counties = uscities[uscities['STATE'] == 'Texas']['COUNTY'].tolist()
texas_cities = uscities[uscities['STATE'] == 'Texas'][['COUNTY', 'CITY']].values
auto_makes = auto_make_model['AUTO_MAKE'].tolist()
auto_models_dict = auto_make_model.groupby('AUTO_MAKE')['AUTO_MODEL'].apply(list).to_dict()

# Generate synthetic data
records = []
for year in range(2015, 2025):
    policy_year = auto_insurance_policy_master[auto_insurance_policy_master['POLICY_BIND_DATE'].str.startswith(str(year))]
    num_records = len(policy_year) + int(len(policy_year) * random.uniform(0.63, 0.83))
    
    application_dates = random_date(start_date, end_date, num_records)
    application_ids = [f"APP{str(i).zfill(8)}" for i in range(10000000, 10000000 + num_records)]
    prospect_ids = [f"PR{str(i).zfill(8)}" for i in range(10000000, 10000000 + num_records)]
    customer_ids = [f"CUST{str(i).zfill(8)}" for i in range(10000000, 10000000 + num_records)]
    
    agents = np.random.choice(agent_ids, num_records)
    counties = np.random.choice(texas_counties, num_records)
    cities = [random.choice(texas_cities[texas_cities[:, 0] == county][:, 1]) for county in counties]
    makes = np.random.choice(auto_makes, num_records)
    models = [random.choice(auto_models_dict[make]) for make in makes]
    
    expected_csls = np.random.choice(["100/300", "250/500", "500/1000"], num_records, p=[0.47, 0.38, 0.15])
    deductibles = np.random.choice([250, 500, 1000, 2000], num_records, p=[0.15, 0.45, 0.30, 0.10])
    
    base_premiums = np.random.randint(750, 2500, num_records)
    cover_premiums = [sum(np.random.randint(50, 350) for _ in range(np.random.randint(1, 8))) for _ in range(num_records)]
    safety_scores = np.random.uniform(0.15, 0.97, num_records)
    safety_premiums = safety_scores * 1000
    umbrella_premiums = np.random.choice([250, 500, 750, 1000], num_records)
    
    expected_premiums = base_premiums + cover_premiums - safety_premiums + umbrella_premiums
    expected_premiums[expected_premiums <= 0] = 750
    
    quote_expected_by_dates = application_dates + pd.to_timedelta(np.random.randint(3, 10, num_records), unit='D')
    competition_levels = np.random.choice(["Low Competition", "Medium Competition", "High Competition"], num_records, p=[0.27, 0.57, 0.16])
    
    past_three_yrs_no_claims = np.random.choice(["0", "1", "2", "3 and above"], num_records, p=[0.21, 0.17, 0.45, 0.17])
    csl_values = np.array([int(csl.split('/')[0]) * 100000 for csl in expected_csls])
    
    past_three_yrs_claim_amts = np.zeros(num_records)
    past_three_yrs_claim_amts[past_three_yrs_no_claims == "1"] = np.random.randint(0, 0.45 * csl_values[past_three_yrs_no_claims == "1"])
    past_three_yrs_claim_amts[past_three_yrs_no_claims == "2"] = np.random.randint(0.25 * csl_values[past_three_yrs_no_claims == "2"], 0.65 * csl_values[past_three_yrs_no_claims == "2"])
    past_three_yrs_claim_amts[past_three_yrs_no_claims == "3 and above"] = np.random.randint(0.65 * csl_values[past_three_yrs_no_claims == "3 and above"], csl_values[past_three_yrs_no_claims == "3 and above"])
    
    no_of_drivers = np.random.choice([1, 2, 3, 4], num_records, p=[0.70, 0.20, 0.07, 0.03])
    
    comprehensive_covers = np.random.choice(["YES", "NO"], num_records, p=[0.8546, 0.1454])
    collision_covers = np.random.choice(["YES", "NO"], num_records, p=[0.6788, 0.3212])
    uninsured_motorists = np.random.choice(["YES", "NO"], num_records, p=[0.4367, 0.5633])
    medical_benefits = np.random.choice(["YES", "NO"], num_records, p=[0.2389, 0.7611])
    pip_covers = np.random.choice(["YES", "NO"], num_records, p=[0.6721, 0.3279])
    accident_forgiveness = np.random.choice(["YES", "NO"], num_records, p=[0.2468, 0.7532])
    roadside_assistances = np.random.choice(["YES", "NO"], num_records, p=[0.7432, 0.2568])
    loss_of_uses = np.random.choice(["YES", "NO"], num_records, p=[0.2913, 0.7087])
    anti_theft_devices = np.random.choice(["YES", "NO"], num_records, p=[0.2913, 0.7087])
    
    safety_scores = np.random.choice([None, *np.random.uniform(0.15, 0.97, num_records)], num_records, p=[0.78, *np.repeat(0.22 / num_records, num_records)])
    
    records.append(pd.DataFrame({
        "APPLICATION_ID": application_ids,
        "APPLICATION_DATE": application_dates,
        "QUOTE_CONVERTED": "Quote not converted",
        "POLICY_NUMBER": None,
        "CUSTOMER_ID": None,
        "POLICY_PREMIUM": None,
        "POLICY_DEDUCTIBLE": None,
        "POLICY_CSL": None,
        "POLICY_BIND_DATE": None,
        "AGENT_ID": agents,
        "PROSPECT_ID": prospect_ids,
        "PROSPECT_STATE": "Texas",
        "PROSPECT_COUNTY": counties,
        "PROSPECT_CITY": cities,
        "AUTO_MAKE": makes,
        "AUTO_MODEL": models,
        "EXPECTED_CSL": expected_csls,
        "DEDUCTIBLE_LOOKED_FOR": deductibles,
        "EXPECTED_PREMIUM": expected_premiums,
        "QUOTE_EXPECTED_BY_DATE": quote_expected_by_dates,
        "COMPETITION_LEVEL": competition_levels,
        "PAST_THREE_YRS_NO_CLAIMS": past_three_yrs_no_claims,
        "PAST_THREE_YRS_CLAIM_AMT": past_three_yrs_claim_amts,
        "NO_OF_DRIVERS": no_of_drivers,
        "COMPREHENSIVE_COVER": comprehensive_covers,
        "COLLISION_COVER": collision_covers,
        "UNINSURED_MOTORIST": uninsured_motorists,
        "MEDICAL_BENEFITS": medical_benefits,
        "PIP_COVER": pip_covers,
        "ACCIDENT_FORGIVENESS": accident_forgiveness,
        "ROADSIDE_ASSISTANCE": roadside_assistances,
        "LOSS_OF_USE": loss_of_uses,
        "ANTI_THEFT_DEVICE": anti_theft_devices,
        "SAFETY_SCORE": safety_scores
    }))

# Concatenate all records into a single DataFrame
synthetic_df = pd.concat(records, ignore_index=True)

In [3]:
synthetic_df.shape

(250948, 35)

In [4]:
synthetic_df.head()

,APPLICATION_ID,APPLICATION_DATE,QUOTE_CONVERTED,POLICY_NUMBER,CUSTOMER_ID,POLICY_PREMIUM,POLICY_DEDUCTIBLE,POLICY_CSL,POLICY_BIND_DATE,AGENT_ID,...,COMPREHENSIVE_COVER,COLLISION_COVER,UNINSURED_MOTORIST,MEDICAL_BENEFITS,PIP_COVER,ACCIDENT_FORGIVENESS,ROADSIDE_ASSISTANCE,LOSS_OF_USE,ANTI_THEFT_DEVICE,SAFETY_SCORE
0,APP10000000,2022-02-07,Quote not converted,None,None,None,None,None,None,AG716547,...,YES,NO,NO,NO,YES,YES,YES,NO,NO,None
1,APP10000001,2016-04-21,Quote not converted,None,None,None,None,None,None,AG717280,...,YES,NO,NO,NO,YES,NO,YES,NO,NO,None
2,APP10000002,2019-12-01,Quote not converted,None,None,None,None,None,None,AG717555,...,YES,YES,NO,NO,NO,NO,YES,NO,NO,None
3,APP10000003,2021-02-20,Quote not converted,None,None,None,None,None,None,AG716264,...,YES,NO,NO,YES,YES,NO,NO,NO,NO,0.22612
4,APP10000004,2015-12-11,Quote not converted,None,None,None,None,None,None,AG716714,...,YES,NO,NO,NO,YES,YES,YES,NO,NO,0.673322


In [5]:
# Filter the rows where POLICY_NUMBER is null
filtered_data = synthetic_df[synthetic_df['POLICY_NUMBER'].isnull()]

# Display the number of rows
num_rows = filtered_data.shape[0]
print(f"Number of rows where POLICY_NUMBER is null: {num_rows}")

# Display the head of the filtered data
filtered_data.head()

Number of rows where POLICY_NUMBER is null: 250948


,APPLICATION_ID,APPLICATION_DATE,QUOTE_CONVERTED,POLICY_NUMBER,CUSTOMER_ID,POLICY_PREMIUM,POLICY_DEDUCTIBLE,POLICY_CSL,POLICY_BIND_DATE,AGENT_ID,...,COMPREHENSIVE_COVER,COLLISION_COVER,UNINSURED_MOTORIST,MEDICAL_BENEFITS,PIP_COVER,ACCIDENT_FORGIVENESS,ROADSIDE_ASSISTANCE,LOSS_OF_USE,ANTI_THEFT_DEVICE,SAFETY_SCORE
0,APP10000000,2022-02-07,Quote not converted,None,None,None,None,None,None,AG716547,...,YES,NO,NO,NO,YES,YES,YES,NO,NO,None
1,APP10000001,2016-04-21,Quote not converted,None,None,None,None,None,None,AG717280,...,YES,NO,NO,NO,YES,NO,YES,NO,NO,None
2,APP10000002,2019-12-01,Quote not converted,None,None,None,None,None,None,AG717555,...,YES,YES,NO,NO,NO,NO,YES,NO,NO,None
3,APP10000003,2021-02-20,Quote not converted,None,None,None,None,None,None,AG716264,...,YES,NO,NO,YES,YES,NO,NO,NO,NO,0.22612
4,APP10000004,2015-12-11,Quote not converted,None,None,None,None,None,None,AG716714,...,YES,NO,NO,NO,YES,YES,YES,NO,NO,0.673322


In [8]:
# Filter the rows where QUOTE_CONVERTED is null
filtered_data = synthetic_df['QUOTE_CONVERTED'].unique()

filtered_data

array(['Quote not converted'], dtype=object)

In [10]:
# Generate new records in a separate dataframe called additional_df
def generate_additional_df(auto_insurance_policy_master):
    num_records = len(auto_insurance_policy_master)
    
    # Generate APPLICATION_ID and PROSPECT_ID
    application_ids = [f'APP{str(i).zfill(8)}' for i in range(1, num_records + 1)]
    prospect_ids = [f'PR{str(i).zfill(8)}' for i in range(1, num_records + 1)]
    
    # Generate QUOTE_EXPECTED_BY_DATE and APPLICATION_DATE
    quote_expected_by_dates = pd.to_datetime(auto_insurance_policy_master['POLICY_BIND_DATE']) - pd.to_timedelta(np.random.randint(0, 4, num_records), unit='D')
    application_dates = quote_expected_by_dates - pd.to_timedelta(np.random.randint(0, 11, num_records), unit='D')
    
    # Generate EXPECTED_PREMIUM
    expected_premiums = auto_insurance_policy_master['POLICY_PREMIUM'] * (1 + np.random.uniform(-0.3, 0.3, num_records))
    
    # Generate COMPETITION_LEVEL
    competition_levels = np.random.choice(['Low Competition', 'Medium Competition', 'High Competition'], num_records,
                                          p=[0.27, 0.57, 0.16])
    
    # Generate PAST_THREE_YRS_NO_CLAIMS
    past_three_yrs_no_claims = np.random.choice(['0', '1', '2', '3 and above'], num_records,
                                                p=[0.21, 0.17, 0.45, 0.17])
    
    # Generate PAST_THREE_YRS_CLAIM_AMT
    csl_values = auto_insurance_policy_master['POLICY_CSL'].str.split('/').str[0].astype(int) * 100000
    past_three_yrs_claim_amt = []
    
    for i in range(num_records):
        if past_three_yrs_no_claims[i] == '0':
            past_three_yrs_claim_amt.append(0)
        elif past_three_yrs_no_claims[i] == '1':
            past_three_yrs_claim_amt.append(np.random.randint(0, int(0.45 * csl_values[i])))
        elif past_three_yrs_no_claims[i] == '2':
            past_three_yrs_claim_amt.append(np.random.randint(int(0.25 * csl_values[i]), int(0.65 * csl_values[i])))
        else:
            past_three_yrs_claim_amt.append(np.random.randint(int(0.65 * csl_values[i]), csl_values[i]))
    
    additional_df = pd.DataFrame({
        'APPLICATION_ID': application_ids,
        'APPLICATION_DATE': application_dates,
        'QUOTE_CONVERTED': ['Quote Converted'] * num_records,
        'POLICY_NUMBER': auto_insurance_policy_master['POLICY_NUMBER'],
        'CUSTOMER_ID': auto_insurance_policy_master['CUSTOMER_ID'],
        'POLICY_PREMIUM': auto_insurance_policy_master['POLICY_PREMIUM'],
        'POLICY_DEDUCTIBLE': auto_insurance_policy_master['POLICY_DEDUCTIBLE'],
        'POLICY_CSL': auto_insurance_policy_master['POLICY_CSL'],
        'POLICY_BIND_DATE': auto_insurance_policy_master['POLICY_BIND_DATE'],
        'AGENT_ID': auto_insurance_policy_master['AGENT_ID'],
        'PROSPECT_ID': prospect_ids,
        'PROSPECT_STATE': ['Texas'] * num_records,
        'PROSPECT_COUNTY': auto_insurance_policy_master['POLICY_COUNTY'],
        'PROSPECT_CITY': auto_insurance_policy_master['POLICY_CITY'],
        'AUTO_MAKE': auto_insurance_policy_master['AUTO_MAKE'],
        'AUTO_MODEL': auto_insurance_policy_master['AUTO_MODEL'],
        'EXPECTED_CSL': auto_insurance_policy_master['POLICY_CSL'],
        'DEDUCTIBLE_LOOKED_FOR': auto_insurance_policy_master['POLICY_DEDUCTIBLE'],
        'EXPECTED_PREMIUM': expected_premiums,
        'QUOTE_EXPECTED_BY_DATE': quote_expected_by_dates,
        'COMPETITION_LEVEL': competition_levels,
        'PAST_THREE_YRS_NO_CLAIMS': past_three_yrs_no_claims,
        'PAST_THREE_YRS_CLAIM_AMT': past_three_yrs_claim_amt,
        # Add remaining columns with relevant data from AUTO_INSURANCE_POLICY_MASTER
        # ...
        # For simplicity in this example we will add only the columns mentioned above
    })
    
    return additional_df

additional_df = generate_additional_df(auto_insurance_policy_master)

In [11]:
additional_df.shape

(143574, 23)

In [12]:
additional_df.head()

,APPLICATION_ID,APPLICATION_DATE,QUOTE_CONVERTED,POLICY_NUMBER,CUSTOMER_ID,POLICY_PREMIUM,POLICY_DEDUCTIBLE,POLICY_CSL,POLICY_BIND_DATE,AGENT_ID,...,PROSPECT_CITY,AUTO_MAKE,AUTO_MODEL,EXPECTED_CSL,DEDUCTIBLE_LOOKED_FOR,EXPECTED_PREMIUM,QUOTE_EXPECTED_BY_DATE,COMPETITION_LEVEL,PAST_THREE_YRS_NO_CLAIMS,PAST_THREE_YRS_CLAIM_AMT
0,APP00000001,2015-09-01,Quote Converted,P71626089,CUST716260,750.0,1000,250/500,2015-09-14,AG716560,...,South Alamo,Subaru,Ascent,250/500,1000,689.276950,2015-09-11,Medium Competition,3 and above,23868111
1,APP00000002,2015-06-14,Quote Converted,P71626090,CUST716261,1838.0,500,100/300,2015-06-18,AG716336,...,Lawn,Saleen,S7 Twin,100/300,500,1346.651121,2015-06-18,High Competition,0,0
2,APP00000003,2015-03-16,Quote Converted,P71626091,CUST716262,750.0,1000,100/300,2015-03-19,AG717150,...,Wallis,Hummer,H2,100/300,1000,617.437913,2015-03-18,Medium Competition,1,15699
3,APP00000004,2015-03-27,Quote Converted,P71626092,CUST716263,2747.0,500,250/500,2015-04-03,AG716845,...,Mansfield,Hennessey,Venom GT,250/500,500,2413.797845,2015-04-03,Medium Competition,1,1428929
4,APP00000005,2015-06-24,Quote Converted,P71626093,CUST716264,2219.0,1000,100/300,2015-07-04,AG716474,...,El Brazil,Daewoo,Matiz,100/300,1000,2518.414908,2015-07-02,High Competition,1,2518531


In [13]:
# Filter the rows where POLICY_NUMBER is null
filtered_data = additional_df[additional_df['POLICY_NUMBER'].isnull()]

# Display the number of rows
num_rows = filtered_data.shape[0]
print(f"Number of rows where POLICY_NUMBER is null: {num_rows}")

# Display the head of the filtered data
filtered_data.head()

Number of rows where POLICY_NUMBER is null: 0


,APPLICATION_ID,APPLICATION_DATE,QUOTE_CONVERTED,POLICY_NUMBER,CUSTOMER_ID,POLICY_PREMIUM,POLICY_DEDUCTIBLE,POLICY_CSL,POLICY_BIND_DATE,AGENT_ID,...,PROSPECT_CITY,AUTO_MAKE,AUTO_MODEL,EXPECTED_CSL,DEDUCTIBLE_LOOKED_FOR,EXPECTED_PREMIUM,QUOTE_EXPECTED_BY_DATE,COMPETITION_LEVEL,PAST_THREE_YRS_NO_CLAIMS,PAST_THREE_YRS_CLAIM_AMT


In [14]:
# Filter the rows where POLICY_NUMBER is NOT null
filtered_data_not_null = additional_df[additional_df['POLICY_NUMBER'].notnull()]

# Display the number of rows
num_rows_not_null = filtered_data_not_null.shape[0]
print(f"Number of rows where POLICY_NUMBER is NOT null: {num_rows_not_null}")

# Display the head of the filtered data
filtered_data_not_null.head()

Number of rows where POLICY_NUMBER is NOT null: 143574


,APPLICATION_ID,APPLICATION_DATE,QUOTE_CONVERTED,POLICY_NUMBER,CUSTOMER_ID,POLICY_PREMIUM,POLICY_DEDUCTIBLE,POLICY_CSL,POLICY_BIND_DATE,AGENT_ID,...,PROSPECT_CITY,AUTO_MAKE,AUTO_MODEL,EXPECTED_CSL,DEDUCTIBLE_LOOKED_FOR,EXPECTED_PREMIUM,QUOTE_EXPECTED_BY_DATE,COMPETITION_LEVEL,PAST_THREE_YRS_NO_CLAIMS,PAST_THREE_YRS_CLAIM_AMT
0,APP00000001,2015-09-01,Quote Converted,P71626089,CUST716260,750.0,1000,250/500,2015-09-14,AG716560,...,South Alamo,Subaru,Ascent,250/500,1000,689.276950,2015-09-11,Medium Competition,3 and above,23868111
1,APP00000002,2015-06-14,Quote Converted,P71626090,CUST716261,1838.0,500,100/300,2015-06-18,AG716336,...,Lawn,Saleen,S7 Twin,100/300,500,1346.651121,2015-06-18,High Competition,0,0
2,APP00000003,2015-03-16,Quote Converted,P71626091,CUST716262,750.0,1000,100/300,2015-03-19,AG717150,...,Wallis,Hummer,H2,100/300,1000,617.437913,2015-03-18,Medium Competition,1,15699
3,APP00000004,2015-03-27,Quote Converted,P71626092,CUST716263,2747.0,500,250/500,2015-04-03,AG716845,...,Mansfield,Hennessey,Venom GT,250/500,500,2413.797845,2015-04-03,Medium Competition,1,1428929
4,APP00000005,2015-06-24,Quote Converted,P71626093,CUST716264,2219.0,1000,100/300,2015-07-04,AG716474,...,El Brazil,Daewoo,Matiz,100/300,1000,2518.414908,2015-07-02,High Competition,1,2518531


In [15]:
# Append the additional records to the existing synthetic data
synthetic_df = pd.concat([synthetic_df, additional_df], ignore_index=True)

In [16]:
synthetic_df.shape

(394522, 35)

In [19]:
# Generate new records in a separate dataframe called df_2025
def generate_df_2025(synthetic_df, auto_insurance_policy_master):
    print("Starting to generate 2025_df...")
    
    # Filter synthetic_df for records with APPLICATION_DATE between 01-December-2024 and 31-December-2024
    synthetic_df['APPLICATION_DATE'] = pd.to_datetime(synthetic_df['APPLICATION_DATE'])
    dec_2024_records = synthetic_df[(synthetic_df['APPLICATION_DATE'] >= '2024-12-01') & (synthetic_df['APPLICATION_DATE'] <= '2024-12-31')]
    
    # Number of records for December 2024 + 14%
    num_records_dec_2024 = len(dec_2024_records)
    num_records_2025 = int(num_records_dec_2024 * 1.14)
    
    print(f"Number of records for December 2024: {num_records_dec_2024}")
    print(f"Number of records for January 2025: {num_records_2025}")
    
    # Generate APPLICATION_ID and PROSPECT_ID
    application_ids = [f'APP{str(i).zfill(8)}' for i in range(num_records_dec_2024 + 1, num_records_dec_2024 + num_records_2025 + 1)]
    prospect_ids = [f'PR{str(i).zfill(8)}' for i in range(num_records_dec_2024 + 1, num_records_dec_2024 + num_records_2025 + 1)]
    print("Generated APPLICATION_ID and PROSPECT_ID")
    
    # Generate APPLICATION_DATE
    application_dates = pd.date_range(start='2025-01-01', end='2025-01-31').to_series().sample(num_records_2025, replace=True).values
    print("Generated APPLICATION_DATE")
    
    # Generate AGENT_ID
    active_agents = auto_insurance_agent_master[auto_insurance_agent_master['AGENT_INACTIVE_FROM'].isnull() | (auto_insurance_agent_master['AGENT_INACTIVE_FROM'] < application_dates[0])]
    agent_ids = active_agents['AGENT_ID'].sample(num_records_2025, replace=True).values
    print("Generated AGENT_ID")
    
    # Generate PROSPECT_COUNTY and PROSPECT_CITY
    prospect_counties = []
    prospect_cities = []
    
    for i in range(num_records_2025):
        if i < num_records_dec_2024:
            prospect_counties.append(auto_insurance_policy_master['POLICY_COUNTY'].iloc[i])
            prospect_cities.append(auto_insurance_policy_master['POLICY_CITY'].iloc[i])
        else:
            random_city_county = uscities[uscities['STATE'] == "Texas"].sample(1)
            prospect_counties.append(random_city_county['COUNTY'].values[0])
            prospect_cities.append(random_city_county['CITY'].values[0])
    print("Generated PROSPECT_COUNTY and PROSPECT_CITY")
    
    # Generate AUTO_MAKE and AUTO_MODEL
    auto_makes = []
    auto_models = []
    
    for i in range(num_records_2025):
        if i < num_records_dec_2024:
            auto_makes.append(auto_insurance_policy_master['AUTO_MAKE'].iloc[i])
            auto_models.append(auto_insurance_policy_master['AUTO_MODEL'].iloc[i])
        else:
            random_auto_make_model = auto_make_model.sample(1)
            auto_makes.append(random_auto_make_model['AUTO_MAKE'].values[0])
            auto_models.append(random_auto_make_model['AUTO_MODEL'].values[0])
    print("Generated AUTO_MAKE and AUTO_MODEL")
    
    # Generate EXPECTED_CSL
    expected_csl_choices = ["100/300", "250/500", "500/1000"]
    
    expected_csls = []
    
    for i in range(num_records_2025):
        if i < num_records_dec_2024:
            expected_csls.append(auto_insurance_policy_master['POLICY_CSL'].iloc[i])
        else:
            expected_csls.append(np.random.choice(expected_csl_choices, p=[0.47, 0.38, 0.15]))
    print("Generated EXPECTED_CSL")
    
    # Generate DEDUCTIBLE_LOOKED_FOR
    deductible_choices = [250, 500, 1000, 2000]
    
    deductible_probs = {
        "100/300": [0.15, 0.45, 0.30, 0.10],
        "250/500": [0.15, 0.45, 0.30, 0.10],
        "500/1000": [0.15, 0.45, 0.30, 0.10]
    }
    
    deductibles_looked_for = []
    
    for i in range(num_records_2025):
        if i < num_records_dec_2024:
            deductibles_looked_for.append(auto_insurance_policy_master['POLICY_DEDUCTIBLE'].iloc[i])
        else:
            csl = expected_csls[i]
            deductibles_looked_for.append(np.random.choice(deductible_choices, p=deductible_probs[csl]))
    print("Generated DEDUCTIBLE_LOOKED_FOR")
    
    # Generate EXPECTED_PREMIUM
    base_premiums = np.random.randint(750, 2501, num_records_2025)
    cover_premiums = np.random.randint(50, 351, num_records_2025)
    safety_premiums = np.random.randint(15000, 97001, num_records_2025)
    umbrella_premiums = np.random.choice([250, 500, 750, 1000], num_records_2025)
    
    expected_premiums = base_premiums + cover_premiums - safety_premiums + umbrella_premiums
    expected_premiums[expected_premiums <= 0] = 750
    print("Generated EXPECTED_PREMIUM")
    
    # Generate QUOTE_EXPECTED_BY_DATE
    quote_expected_by_dates = application_dates + pd.to_timedelta(np.random.randint(3, 11, num_records_2025), unit='D')
    print("Generated QUOTE_EXPECTED_BY_DATE")
    
    # Generate COMPETITION_LEVEL
    competition_levels = np.random.choice(['Low Competition', 'Medium Competition', 'High Competition'], num_records_2025,
                                          p=[0.27, 0.57, 0.16])
    print("Generated COMPETITION_LEVEL")
    
    # Generate PAST_THREE_YRS_NO_CLAIMS
    past_three_yrs_no_claims = np.random.choice(['0', '1', '2', '3 and above'], num_records_2025,
                                                p=[0.21, 0.17, 0.45, 0.17])
    print("Generated PAST_THREE_YRS_NO_CLAIMS")
    
    # Generate PAST_THREE_YRS_CLAIM_AMT
    csl_values = pd.Series(expected_csls).str.split('/').str[0].astype(int) * 100000
    past_three_yrs_claim_amt = []
    
    for i in range(num_records_2025):
        if past_three_yrs_no_claims[i] == '0':
            past_three_yrs_claim_amt.append(0)
        elif past_three_yrs_no_claims[i] == '1':
            past_three_yrs_claim_amt.append(np.random.randint(0, int(0.45 * csl_values[i])))
        elif past_three_yrs_no_claims[i] == '2':
            past_three_yrs_claim_amt.append(np.random.randint(int(0.25 * csl_values[i]), int(0.65 * csl_values[i])))
        else:
            past_three_yrs_claim_amt.append(np.random.randint(int(0.65 * csl_values[i]), csl_values[i]))
    print("Generated PAST_THREE_YRS_CLAIM_AMT")
    
    # Generate NO_OF_DRIVERS
    no_of_drivers = np.random.choice([1, 2, 3, 4], num_records_2025, p=[0.5, 0.3, 0.15, 0.05])
    print("Generated NO_OF_DRIVERS")
    
    # Generate COMPREHENSIVE_COVER
    comprehensive_cover = np.random.choice([True, False], num_records_2025, p=[0.8546, 0.1454])
    print("Generated COMPREHENSIVE_COVER")
    
    # Generate COLLISION_COVER
    collision_cover = np.random.choice([True, False], num_records_2025, p=[0.6788, 0.3212])
    print("Generated COLLISION_COVER")
    
    # Generate UNINSURED_MOTORIST
    uninsured_motorist = np.random.choice([True, False], num_records_2025, p=[0.4367, 0.5633])
    print("Generated UNINSURED_MOTORIST")
    
    # Generate MEDICAL_BENEFITS
    medical_benefits = np.random.choice([True, False], num_records_2025, p=[0.2389, 0.7611])
    print("Generated MEDICAL_BENEFITS")
    
    # Generate PIP_COVER
    pip_cover = np.random.choice([True, False], num_records_2025, p=[0.6721, 0.3279])
    print("Generated PIP_COVER")
    
    # Generate ACCIDENT_FORGIVENESS
    accident_forgiveness = np.random.choice([True, False], num_records_2025, p=[0.2468, 0.7532])
    print("Generated ACCIDENT_FORGIVENESS")
    
    # Generate ROADSIDE_ASSISTANCE
    roadside_assistance = np.random.choice([True, False], num_records_2025, p=[0.7432, 0.2568])
    print("Generated ROADSIDE_ASSISTANCE")
    
    # Generate LOSS_OF_USE
    loss_of_use = np.random.choice([True, False], num_records_2025, p=[0.2913, 0.7087])
    print("Generated LOSS_OF_USE")
    
    # Generate ANTI_THEFT_DEVICE
    anti_theft_device = np.random.choice([True, False], num_records_2025, p=[0.2913, 0.7087])
    print("Generated ANTI_THEFT_DEVICE")
    
    # Generate SAFETY_SCORE
    safety_score = np.random.choice([None, *range(15, 98)], num_records_2025, p=[0.78] + [0.22 / 83] * 83)
    print("Generated SAFETY_SCORE")
    
    # Create the df_2025 dataframe
    df_2025 = pd.DataFrame({
        'APPLICATION_ID': application_ids,
        'APPLICATION_DATE': application_dates,
        'QUOTE_CONVERTED': [None] * num_records_2025,
        'POLICY_NUMBER': [None] * num_records_2025,
        'CUSTOMER_ID': [None] * num_records_2025,
        'POLICY_PREMIUM': [None] * num_records_2025,
        'POLICY_DEDUCTIBLE': [None] * num_records_2025,
        'POLICY_CSL': [None] * num_records_2025,
        'POLICY_BIND_DATE': [None] * num_records_2025,
        'AGENT_ID': agent_ids,
        'PROSPECT_ID': prospect_ids,
        'PROSPECT_STATE': ['Texas'] * num_records_2025,
        'PROSPECT_COUNTY': prospect_counties,
        'PROSPECT_CITY': prospect_cities,
        'AUTO_MAKE': auto_makes,
        'AUTO_MODEL': auto_models,
        'EXPECTED_CSL': expected_csls,
        'DEDUCTIBLE_LOOKED_FOR': deductibles_looked_for,
        'EXPECTED_PREMIUM': expected_premiums,
        'QUOTE_EXPECTED_BY_DATE': quote_expected_by_dates,
        'COMPETITION_LEVEL': competition_levels,
        'PAST_THREE_YRS_NO_CLAIMS': past_three_yrs_no_claims,
        'PAST_THREE_YRS_CLAIM_AMT': past_three_yrs_claim_amt,
        'NO_OF_DRIVERS': no_of_drivers,
        'COMPREHENSIVE_COVER': comprehensive_cover,
        'COLLISION_COVER': collision_cover,
        'UNINSURED_MOTORIST': uninsured_motorist,
        'MEDICAL_BENEFITS': medical_benefits,
        'PIP_COVER': pip_cover,
        'ACCIDENT_FORGIVENESS': accident_forgiveness,
        'ROADSIDE_ASSISTANCE': roadside_assistance,
        'LOSS_OF_USE': loss_of_use,
        'ANTI_THEFT_DEVICE': anti_theft_device,
        'SAFETY_SCORE': safety_score,
        'QUOTE_DATE': [None] * num_records_2025,
        'QUOTE_ACCEPTANCE_DATE': [None] * num_records_2025,
        'LOSS_REASON': [None] * num_records_2025
    })
    
    print("Completed generating 2025_df")
    return df_2025

df_2025 = generate_df_2025(synthetic_df, auto_insurance_policy_master)

Starting to generate 2025_df...
Number of records for December 2024: 2773
Number of records for January 2025: 3161
Generated APPLICATION_ID and PROSPECT_ID
Generated APPLICATION_DATE
Generated AGENT_ID
Generated PROSPECT_COUNTY and PROSPECT_CITY
Generated AUTO_MAKE and AUTO_MODEL
Generated EXPECTED_CSL
Generated DEDUCTIBLE_LOOKED_FOR
Generated EXPECTED_PREMIUM
Generated QUOTE_EXPECTED_BY_DATE
Generated COMPETITION_LEVEL
Generated PAST_THREE_YRS_NO_CLAIMS
Generated PAST_THREE_YRS_CLAIM_AMT
Generated NO_OF_DRIVERS
Generated COMPREHENSIVE_COVER
Generated COLLISION_COVER
Generated UNINSURED_MOTORIST
Generated MEDICAL_BENEFITS
Generated PIP_COVER
Generated ACCIDENT_FORGIVENESS
Generated ROADSIDE_ASSISTANCE
Generated LOSS_OF_USE
Generated ANTI_THEFT_DEVICE
Generated SAFETY_SCORE
Completed generating 2025_df


In [20]:
# Filter the rows where QUOTE_CONVERTED is null
filtered_data = df_2025[df_2025['QUOTE_CONVERTED'].isnull()]

# Display the number of rows
num_rows = filtered_data.shape[0]
print(f"Number of rows where QUOTE_CONVERTED is null: {num_rows}")

# Display the head of the filtered data
filtered_data.head()

Number of rows where QUOTE_CONVERTED is null: 3161


,APPLICATION_ID,APPLICATION_DATE,QUOTE_CONVERTED,POLICY_NUMBER,CUSTOMER_ID,POLICY_PREMIUM,POLICY_DEDUCTIBLE,POLICY_CSL,POLICY_BIND_DATE,AGENT_ID,...,MEDICAL_BENEFITS,PIP_COVER,ACCIDENT_FORGIVENESS,ROADSIDE_ASSISTANCE,LOSS_OF_USE,ANTI_THEFT_DEVICE,SAFETY_SCORE,QUOTE_DATE,QUOTE_ACCEPTANCE_DATE,LOSS_REASON
0,APP00002774,2025-01-03,None,None,None,None,None,None,None,AG716768,...,False,False,False,False,False,False,None,None,None,None
1,APP00002775,2025-01-31,None,None,None,None,None,None,None,AG716483,...,False,False,False,False,False,False,None,None,None,None
2,APP00002776,2025-01-22,None,None,None,None,None,None,None,AG716684,...,False,True,False,False,False,False,None,None,None,None
3,APP00002777,2025-01-27,None,None,None,None,None,None,None,AG717516,...,True,True,False,True,False,False,21,None,None,None
4,APP00002778,2025-01-27,None,None,None,None,None,None,None,AG717554,...,False,True,False,False,True,False,None,None,None,None


In [21]:
# Filter the rows where QUOTE_CONVERTED is NOT null
filtered_data_not_null = df_2025[df_2025['QUOTE_CONVERTED'].notnull()]

# Display the number of rows
num_rows_not_null = filtered_data_not_null.shape[0]
print(f"Number of rows where QUOTE_CONVERTED is NOT null: {num_rows_not_null}")

# Display the head of the filtered data
filtered_data_not_null.head()

Number of rows where QUOTE_CONVERTED is NOT null: 0


,APPLICATION_ID,APPLICATION_DATE,QUOTE_CONVERTED,POLICY_NUMBER,CUSTOMER_ID,POLICY_PREMIUM,POLICY_DEDUCTIBLE,POLICY_CSL,POLICY_BIND_DATE,AGENT_ID,...,MEDICAL_BENEFITS,PIP_COVER,ACCIDENT_FORGIVENESS,ROADSIDE_ASSISTANCE,LOSS_OF_USE,ANTI_THEFT_DEVICE,SAFETY_SCORE,QUOTE_DATE,QUOTE_ACCEPTANCE_DATE,LOSS_REASON


In [22]:
# Append the df_2025 records to the existing synthetic data
synthetic_df = pd.concat([synthetic_df, df_2025], ignore_index=True)

In [23]:
# Save the synthetic data to the specified file path
synthetic_df.to_csv('C:/Users/10741867/OneDrive - LTIMindtree/Documents/Datasets/Solution Datasets/Solution 1 - PNC Quote Conversion and Prioritization/Datasets/Regenerated Datasets/AUTO_INSURANCE_QUOTE_CONVERSION_MASTER.csv', index=False)